%md
## 클래스룸 설정

다음 셀을 실행하여 이 코스에 필요한 작업 환경을 구성합니다.

In [0]:
####################################################################################
# 카탈로그, 스키마, 볼륨 이름에 대한 Python 변수 설정 (필요한 경우 변경)
catalog_name = "magazon"
schema_name = "ingestion_lab"
volume_name = "myfiles"
####################################################################################

####################################################################################
# 카탈로그, 스키마, 볼륨이 존재하지 않으면 생성
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")
####################################################################################

####################################################################################
# 지정된 catalog.schema.volume에 employees.csv라는 파일 생성
import pandas as pd
data = [
    ["1111", "Kristi", "USA", "Manager"],
    ["2222", "Sophia", "Greece", "Developer"],
    ["3333", "Peter", "USA", "Developer"],
    ["4444", "Zebi", "Pakistan", "Administrator"]
]
columns = ["ID", "Firstname", "Country", "Role"] 
df = pd.DataFrame(data, columns=columns)
file_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/employees.csv"
df.to_csv(file_path, index=False)
################################################################################

####################################################################################
# 지정된 catalog.schema.volume에 taxi_files라는 볼륨 생성
spark.sql(f'CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.taxi_files')
output_volume = f'/Volumes/{catalog_name}/{schema_name}/taxi_files'

sdf = spark.table("samples.nyctaxi.trips")

(sdf
    .write
    .mode("overwrite")
    .csv(output_volume, header=True)
    )
####################################################################################

## 실습 시작

1. 현재 카탈로그를 **magazon**으로, 스키마를 **ingesting_data**로 설정합니다.

    **힌트**:
    - 카탈로그: `USE CATALOG`
    - 스키마: `USE SCHEMA`

In [0]:
%sql
USE CATALOG magazon;
USE SCHEMA ingestion_lab;

2. 현재 카탈로그와 스키마를 보려면 쿼리를 실행합니다. 결과가 **magazon** 카탈로그와 **ingestion_lab** 스키마를 표시하는지 확인합니다.

In [0]:

%sql
SELECT current_catalog() AS catalog, current_schema() AS schema;

3. 스키마에서 사용 가능한 볼륨을 확인하고 **taxi_files** 볼륨이 목록에 있는지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

4. **taxi_files** 볼륨의 파일을 나열하고 **name** 열을 확인하여 볼륨에 저장된 파일 유형을 확인합니다. 밑줄(_)로 시작하는 추가 파일은 무시합니다.

**힌트**: 볼륨에 액세스하려면 다음 경로 형식을 사용합니다: */Volumes/catalog_name/schema_name/volume_name/*.

In [0]:
%sql
-- TODO
-- <FILL_IN>

5. 볼륨 경로를 직접 쿼리하고 적절한 파일 형식을 사용하여 파일의 데이터를 미리 보기합니다. 볼륨 경로를 백틱(`)으로 묶어야 합니다.

**힌트**: SELECT * FROM \<file-format\>.\`\<path-to-volume-taxi_files\>\`

In [0]:
%sql
-- TODO
-- <FILL_IN>

6. 스키마에 다음 열을 포함하는 **taxitrips_bronze**라는 테이블을 생성합니다:
| 필드 이름 | 필드 타입 |
| --- | --- |
| tpep_pickup_datetime | TIMESTAMP |
| tpep_dropoff_datetime | TIMESTAMP |
| trip_distance | DOUBLE |
| fare_amount | DOUBLE |
| pickup_zip | INT |
| dropoff_zip | INT |

**참고:** DROP TABLE 문은 오류를 방지하기 위해 테이블이 이미 존재하는 경우 삭제합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

7. [COPY INTO](https://docs.databricks.com/en/sql/language-manual/delta-copy-into.html) 문을 사용하여 **taxi_files** 볼륨의 파일을 **taxitrips_bronze** 테이블에 로드합니다. 다음 옵션을 포함합니다:
    - FROM `path-to-taxi_files`
    - FILEFORMAT = '\<file-format\>'
    - FORMAT_OPTIONS
      - 'header' = 'true'
      - 'inferSchema' = 'true'

    21,932개의 행이 삽입되었는지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

8. **taxitrips_bronze** 테이블의 행 수를 세어봅니다. 테이블에 21,932개의 행이 있는지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

9. **taxitrips_bronze** 테이블의 히스토리를 확인합니다. 버전 0과 버전 1이 사용 가능한지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

10. 다음 스크립트를 실행하여 **trip_distance**가 *1*보다 작은 모든 행을 삭제합니다. *5,387*개의 행이 삭제되었는지 확인합니다.

In [0]:
%sql
DELETE FROM taxitrips_bronze
  WHERE trip_distance < 1;

11. **taxitrips_bronze** 테이블의 히스토리를 확인합니다. **operation** 열을 확인합니다. *DELETE* 작업이 발생한 버전을 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

12. 쿼리를 실행하여 **taxitrips_bronze** 테이블의 현재 버전에 있는 총 행 수를 세어봅니다. 현재 테이블에 *16,545*개의 행이 있는지 확인합니다.

**힌트:** 기본적으로 가장 최근 버전이 사용됩니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

13. 테이블의 원래 버전을 쿼리하여 처음 생성되었을 때의 행 수를 세어봅니다. 원래 테이블에 *21,932*개의 행이 있는지 확인합니다.

**힌트:** FROM \<table> VERSION AS OF \<n>

In [0]:
%sql
-- TODO
-- <FILL_IN>

14. 아차! 실수로 이전에 행을 삭제하면 안 되는 것이었습니다. [RESTORE](https://docs.databricks.com/en/sql/language-manual/delta-restore.html) 문을 사용하여 Delta 테이블을 *DELETE* 작업 이전의 원래 상태로 복원합니다.

In [0]:
%sql
-- TODO: RESTORE TABLE 구문을 사용하여 버전 1번의 데이터를 복구하세요
-- <FILL_IN>

15. **taxitrips_bronze** 테이블의 히스토리를 확인합니다. 가장 최근 버전에 **operation** *RESTORE*가 포함되어 있는지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

16. 현재 **taxitrips_bronze** 테이블의 총 행 수를 세어봅니다. 테이블의 가장 최근 버전에 *21,932*개의 행이 있는지 확인합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

17. **ingestion_lab** 스키마를 삭제합니다.

In [0]:
%sql
-- TODO
-- <FILL_IN>

### 요약
이 실습을 완료하면 여러분은 이제 다음을 편안하게 수행할 수 있을 것입니다:
* 표준 Delta Lake 테이블 생성 및 데이터 조작 명령 완료
* 테이블 히스토리를 포함한 테이블 메타데이터 검토
* 스냅샷 쿼리 및 롤백을 위한 Delta Lake 버전 관리 활용